In [136]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [137]:
df = pd.read_csv("/content/healthcare-dataset-stroke-data.csv")

In [138]:
df.head(5)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [119]:
df.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


In [139]:
df.drop(columns=['id'],axis=1,inplace=True)

In [140]:
from numpy.random.mtrand import random
x_train,x_test,y_train,y_test = train_test_split(df.drop(columns=["stroke"]),df["stroke"],test_size=0.2,random_state=42,stratify=df['stroke'])

In [141]:
x_train.sample(3)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
2965,Male,79.00,1,0,No,Private,Urban,96.52,21.7,Unknown
3251,Male,0.56,0,0,No,children,Rural,127.23,20.1,Unknown
2845,Female,5.00,0,0,No,children,Rural,93.03,16.3,Unknown


In [142]:
trf1 = ColumnTransformer([
    ("impute_bmi",SimpleImputer(),[8])
],remainder="passthrough"
)

In [149]:
trf2 = ColumnTransformer([
    ("ohe_gender_ever_married_work_type_Residence_type_smoking_status",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[1,5,6,7,9])
],remainder="passthrough"
)

In [150]:
trf3 = ColumnTransformer([
    ("scale_age_avg_glucose_bmi", StandardScaler(), [16, 17, 20])
], remainder="passthrough")

In [151]:
trf4 = LogisticRegression(class_weight="balanced")

In [152]:
pipe = Pipeline([
    ("trf1",trf1),
    ("trf2",trf2),
    ("trf3",trf3),
    ("trf4",trf4)
])

In [153]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_bmi', SimpleImputer(),
                                                  [8])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_gender_ever_married_work_type_Residence_type_smoking_status',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 5, 6, 7, 9])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale_age_avg_glucose_bmi',
                                                  StandardScaler(),
                                                  [16, 17, 20])])),
                ('trf4', LogisticRegression(class_weight='balanced'))])

In [154]:
y_pred = pipe.predict(x_test)

In [155]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(1022,))

In [156]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7465753424657534

In [158]:
import pickle
pickle.dump(pipe,open("pipe.pkl","wb"))

In [159]:
y_pred = pipe.predict(x_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

accuracy_score(y_test, y_pred)
confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred))
roc_auc_score(y_test, pipe.predict_proba(x_test)[:, 1])

              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022



np.float64(0.843724279835391)